In [1]:
#using Pkg
#Pkg.add("JSON")
#Pkg.add("DotEnv")
#Pkg.add("HTTP")
#Pkg.add("JSON")
#Pkg.add("URIs")
#Pkg.add("ConfigEnv")

using DotEnv, HTTP, URIs,JSON


This is a modified version of the get_data program in the api documentation.  Designed here to illustrate error correction methods

In [2]:
function get_data(property) 
    cfg = DotEnv.config()
    #a bogus token which works for public request
    default_token="eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiJ9.eyJpZCI6IjhjNzRhMjY0YjE0Yj"
    if isempty(cfg)
        response = Dict(
            "error" => "configuration",
            "error_description" => ".env files doesn't exist or is empty"
            )
        return JSON.json(response)
    elseif cfg["api_url"] == ""
        response = Dict(
            "error" => "api_url",
            "error_description" => "there is no api_url, can't return output"
            )
        return JSON.json(response)
    elseif cfg["access_token"] == ""
        #assuming this is a public request
        cfg["access_token"] = default_token
    end
                
    api_url = string(cfg["api_url"],"/",property)
    new_headers = Dict(
        "Accept" => "application/json",
        "Authorization" => string("Bearer ",cfg["access_token"]),
    )
    response = HTTP.request("GET", api_url, new_headers)
        
    r = String(response.body)
    check = JSON.parse(r)
    if "error" in keys(check)
        println("detected error")
    end
    return r
end

get_data (generic function with 1 method)

In [3]:

placements = get_data("from_metadata");


In [4]:
md = JSON.parse(placements);

```
for m in md
    if n < 1
        current = Dict(["fname" => m["fname"], "lname" => m["lname"]])
        println("........")
        for(key, value) in m
            println(key, " => ", value)
        end
        println("........")
    end
    n += 1
end
```

In [35]:
n = 0
aid = 0
applicants = Set()
duplicates = Set()
for m in md
    current = Dict(["fname" => m["fname"], "lname" => m["lname"]])
    for a in applicants
        if a["record"] == current
            push!(duplicates,Dict(["current" => m["aid"], "original" => a["aid"]]))
            break
        end
    end
    push!(applicants, Dict(["aid" => m["aid"], "record" => current]))
end


In [36]:
println(length(applicants))
println(length(duplicates))
println(duplicates)

35505
3767
Set(Any[Dict("original" => "17966", "current" => "15947"), Dict("original" => "41620", "current" => "16088"), Dict("original" => "31920", "current" => "29086"), Dict("original" => "20973", "current" => "14193"), Dict("original" => "22513", "current" => "3318"), Dict("original" => "32199", "current" => "18163"), Dict("original" => "21811", "current" => "10370"), Dict("original" => "25231", "current" => "16952"), Dict("original" => "39808", "current" => "13417"), Dict("original" => "16802", "current" => "16262"), Dict("original" => "51975", "current" => "5964"), Dict("original" => "31256", "current" => "24942"), Dict("original" => "5425", "current" => "1803"), Dict("original" => "6956", "current" => "6731"), Dict("original" => "7117", "current" => "27603"), Dict("original" => "25727", "current" => "2867"), Dict("original" => "7557", "current" => "15254"), Dict("original" => "5247", "current" => "6071"), Dict("original" => "30549", "current" => "12036"), Dict("original" => "261

To flag the entries I use an elementary method.  First I'll create an empty set to store the placements I have already checked called `records` and another set to hold the aids I want to flag called `flags`.

Then I'll iterate through the placements one by one and check whether each placement is already in `records`.  If it is there already I'll try to add the aid to `flags`.  If it isn't there I'll add the record to `records`

In [30]:
records = Set()
push!(records,Dict(["aid" => aid, "record" => test]))
#flags = Set()
#x = [1,2,3]
#push!(s,[1,2,3])
println(records)

Set(Any[Dict{String, Any}("record" => Dict("lname" => "Zhang", "fname" => "Wan"), "aid" => "59156")])


In [7]:
for placement in placements
    t = [placement["aid"],placement["to_oid"],placement["from_oid"],placement["year"],placement["postype"]]
    #println( "checking ", t)
    if t in records
        push!(flags,placement["aid"])
    else
        push!(records,t)
    end 
end 
        

LoadError: MethodError: no method matching getindex(::Char, ::String)
[0mClosest candidates are:
[0m  getindex(::AbstractChar) at ~/julia/share/julia/base/char.jl:202
[0m  getindex(::AbstractChar, [91m::Integer[39m) at ~/julia/share/julia/base/char.jl:203
[0m  getindex(::AbstractChar, [91m::Integer...[39m) at ~/julia/share/julia/base/char.jl:204
[0m  ...

In [8]:
println(length(records))
println(length(flags))

0
0


In [9]:
println(flags)

Set{Any}()
